# Wav2Vec2 Audio Emotion Training
Fine-tune a pretrained Wav2Vec2 model for speech emotion classification and save the processor and model artifacts.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os

import evaluate
import librosa
import numpy as np
import pandas as pd
import torch
from datasets import Audio, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoProcessor,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    Wav2Vec2ForSequenceClassification
)

DATASET_DIR = Path("../audio")
OUTPUT_DIR = Path("../models/audio_emotion_model")
MODEL_NAME = "facebook/wav2vec2-base"
TARGET_SAMPLING_RATE = 16000
SEED = 42

In [ ]:
def collect_audio_samples(dataset_dir: Path):
    records = []
    for audio_path in sorted(dataset_dir.rglob("*.wav")):
        label = audio_path.parent.name
        records.append({"path": str(audio_path), "label": label})
    return pd.DataFrame(records)

metadata_df = collect_audio_samples(DATASET_DIR)
if metadata_df.empty:
    raise ValueError(f"No WAV files found under {DATASET_DIR.resolve()}")

label_encoder = LabelEncoder()
metadata_df["label_id"] = label_encoder.fit_transform(metadata_df["label"])

train_df, eval_df = train_test_split(
    metadata_df,
    test_size=0.2,
    random_state=SEED,
    stratify=metadata_df["label_id"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "eval": Dataset.from_pandas(eval_df.reset_index(drop=True))
})

dataset = dataset.cast_column("path", Audio(sampling_rate=TARGET_SAMPLING_RATE))
dataset

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
accuracy_metric = evaluate.load("accuracy")

def preprocess_batch(batch):
    audio = batch["path"]
    signal = np.asarray(audio["array"], dtype=np.float32)
    if audio["sampling_rate"] != TARGET_SAMPLING_RATE:
        signal = librosa.resample(signal, orig_sr=audio["sampling_rate"], target_sr=TARGET_SAMPLING_RATE)

    processed = processor(
        signal,
        sampling_rate=TARGET_SAMPLING_RATE,
        return_attention_mask=True
    )

    batch["input_values"] = processed["input_values"][0]
    batch["attention_mask"] = processed["attention_mask"][0]
    batch["labels"] = int(batch["label_id"])
    return batch

encoded_dataset = dataset.map(
    preprocess_batch,
    remove_columns=dataset["train"].column_names
)
encoded_dataset

In [ ]:
@dataclass
class AudioEmotionCollator:
    processor: AutoProcessor

    def __call__(self, features):
        input_features = [
            {
                "input_values": feature["input_values"],
                "attention_mask": feature["attention_mask"]
            }
            for feature in features
        ]
        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")
        batch["labels"] = torch.tensor([feature["labels"] for feature in features], dtype=torch.long)
        return batch

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    label2id={label: int(index) for index, label in enumerate(label_encoder.classes_)},
    id2label={int(index): label for index, label in enumerate(label_encoder.classes_)},
    classifier_proj_size=256
)
model.freeze_feature_encoder()

data_collator = AudioEmotionCollator(processor=processor)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["eval"],
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))

print(f"Saved fine-tuned model and processor to {OUTPUT_DIR}")